In [18]:
import json
import shutil
from pathlib import Path
from shapely.wkt import loads as load_wkt
from random import Random

In [19]:
base_dir = Path(r"C:\Users\misha\OneDrive\Desktop\DL")
source_train_dir = base_dir / "train"
images_dir = source_train_dir / "images"
labels_dir = source_train_dir / "labels"

out_root = base_dir / "yolo-damage" / "data"

# Маппинг классов
damage_map = {
    "no-damage": 0,
    "minor-damage": 1,
    "major-damage": 2,
    "destroyed": 3,
}

In [20]:
for split in ["train", "val"]:
    (out_root / split / "images").mkdir(parents=True, exist_ok=True)
    (out_root / split / "labels").mkdir(parents=True, exist_ok=True)

In [21]:
def poly_to_yolo(wkt_str, img_width=1024, img_height=1024):
    poly = load_wkt(wkt_str)
    minx, miny, maxx, maxy = poly.bounds

    xc = ((minx + maxx) / 2) / img_width
    yc = ((miny + maxy) / 2) / img_height
    bw = (maxx - minx) / img_width
    bh = (maxy - miny) / img_height
    return xc, yc, bw, bh

In [22]:
def convert_json_file(json_path, images_dir, out_img_dir, out_lbl_dir):
    if "post_disaster" not in json_path.name:
        return 0

    with open(json_path) as f:
        data = json.load(f)

    img_name = json_path.name.replace(".json", ".png")
    img_path = images_dir / img_name

    if not img_path.exists():
        return 0

    lines = []
    for feat in data["features"]["xy"]:
        props = feat["properties"]
        subtype = props.get("subtype")
        
        if subtype not in damage_map:
            continue

        wkt_str = feat["wkt"]
        xc, yc, bw, bh = poly_to_yolo(wkt_str)
        
        if bw <= 0 or bh <= 0:
            continue

        cls = damage_map[subtype]
        lines.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    if not lines:
        return 0

    shutil.copy2(img_path, out_img_dir / img_path.name)
    (out_lbl_dir / f"{json_path.stem}.txt").write_text("\n".join(lines))
    return len(lines)

In [23]:
json_files = sorted(labels_dir.glob("*.json"))
count = 0

for jp in json_files:
    count += convert_json_file(
        jp,
        images_dir,
        out_root / "train" / "images",
        out_root / "train" / "labels",
    )

print(f"Total boxes written: {count}")

Total boxes written: 159794


In [24]:
rng = Random(42)
train_img_dir = out_root / "train" / "images"
train_lbl_dir = out_root / "train" / "labels"

train_imgs = sorted(train_img_dir.glob("*.png"))
val_count = max(1, len(train_imgs) // 10)
val_imgs = set(rng.sample(train_imgs, val_count))

val_img_dir = out_root / "val" / "images"
val_lbl_dir = out_root / "val" / "labels"

for img_path in val_imgs:
    lbl_path = train_lbl_dir / f"{img_path.stem}.txt"
    # Перемещаем случайно выбранные файлы в папку val
    shutil.move(str(img_path), str(val_img_dir / img_path.name))
    shutil.move(str(lbl_path), str(val_lbl_dir / lbl_path.name))

In [25]:
for split in ["train", "val"]:
    img_dir = out_root / split / "images"
    lbl_dir = out_root / split / "labels"
    print(f"{split} | Images: {len(list(img_dir.glob('*.png')))} | Labels: {len(list(lbl_dir.glob('*.txt')))}")

train | Images: 2017 | Labels: 2017
val | Images: 224 | Labels: 224


In [26]:
from pathlib import Path

project_path = Path(r"C:\Users\misha\OneDrive\Desktop\DL\yolo-damage")
yaml_path = project_path / "data.yaml"

yaml_content = """path: C:/Users/misha/OneDrive/Desktop/DL/yolo-damage/data
train: train/images
val: val/images

names:
  0: no-damage
  1: minor-damage
  2: major-damage
  3: destroyed
"""

yaml_path.write_text(yaml_content, encoding="utf-8")

print(f"File created at: {yaml_path}")
print(f"data.yaml exists: {yaml_path.exists()}")

File created at: C:\Users\misha\OneDrive\Desktop\DL\yolo-damage\data.yaml
data.yaml exists: True
